# ⚡ Ultra-Fast PDF Chat - Multiple Model Options

## 🚀 Speed Options
1. **Flan-T5 Large** (FASTEST - 1-2 seconds) ⭐ Recommended for Kaggle
2. **Mistral-7B** (Better quality - 3-5 seconds)

## 📊 Expected Performance
- **Flan-T5:** 1-2 sec per query, 2GB GPU
- **Mistral-7B:** 3-5 sec per query, 4GB GPU

This notebook uses **Flan-T5 by default** for maximum speed!

In [ ]:
# 🎯 CHOOSE YOUR MODEL HERE
MODEL_CHOICE = 'flan-t5'  # Options: 'flan-t5' (fast) or 'mistral' (better)

print(f"📌 Selected model: {MODEL_CHOICE.upper()}")
if MODEL_CHOICE == 'flan-t5':
    print("   Speed: ⚡⚡⚡ Ultra-fast (1-2 sec)")
    print("   Quality: ⭐⭐⭐ Good")
    print("   GPU RAM: 2GB")
else:
    print("   Speed: ⚡⚡ Fast (3-5 sec)")
    print("   Quality: ⭐⭐⭐⭐ Better")
    print("   GPU RAM: 4GB")

In [ ]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

print("📦 Installing packages (30-60 sec)...\n")
print("ℹ️  Note: transformers, torch, accelerate, gradio are pre-installed on Kaggle\n")

# Only install packages NOT pre-installed on Kaggle
# This is 4-6x FASTER than reinstalling everything!
!pip install -q sentence-transformers chromadb pdfplumber

if MODEL_CHOICE == 'mistral':
    !pip install -q bitsandbytes

print("\n✅ Ready! (Much faster than before ⚡)")

In [ ]:
import sys
import logging
logging.getLogger('transformers').setLevel(logging.ERROR)

import torch
import pdfplumber
from pathlib import Path
from typing import Dict, List
import numpy as np
import time
import uuid

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import gradio as gr

print(f"✅ Libraries loaded")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPUs: {torch.cuda.device_count()}")

In [ ]:
class PDFProcessor:
    def extract_text(self, pdf_path: str) -> Dict:
        print(f"📖 {Path(pdf_path).name}...", end=" ", flush=True)
        text_by_page = {}
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, 1):
                    if text := page.extract_text():
                        text_by_page[page_num] = text
            print(f"{len(text_by_page)} pages ✓")
            return {
                'file_name': Path(pdf_path).name,
                'num_pages': len(text_by_page),
                'text_by_page': text_by_page
            }
        except Exception as e:
            print(f"Error: {e}")
            return None

class TextChunker:
    def __init__(self, chunk_size=600, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def chunk_with_metadata(self, text_by_page: Dict, file_name: str) -> List[Dict]:
        chunks = []
        for page_num, text in text_by_page.items():
            start = 0
            while start < len(text):
                end = start + self.chunk_size
                chunk = text[start:end].strip()
                if chunk:
                    chunks.append({
                        'text': chunk,
                        'page': page_num,
                        'file_name': file_name
                    })
                start = end - self.chunk_overlap
        return chunks

class VectorStore:
    def __init__(self):
        self.client = chromadb.Client(Settings(anonymized_telemetry=False))
        try:
            self.collection = self.client.get_collection('pdfs')
        except:
            self.collection = self.client.create_collection('pdfs', metadata={"hnsw:space": "cosine"})
    
    def add_documents(self, chunks: List[Dict], embeddings: np.ndarray):
        self.collection.add(
            ids=[str(uuid.uuid4()) for _ in chunks],
            embeddings=embeddings.tolist(),
            documents=[c['text'] for c in chunks],
            metadatas=[{'page': c['page'], 'file_name': c['file_name']} for c in chunks]
        )
    
    def query(self, emb: np.ndarray, top_k=3):
        results = self.collection.query(query_embeddings=[emb.tolist()], n_results=top_k)
        return {
            'documents': results['documents'][0] if results['documents'] else [],
            'metadatas': results['metadatas'][0] if results['metadatas'] else []
        }

print("✅ Core classes ready")

In [ ]:
print("📥 Loading embedding model...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"✅ Embeddings ready on {device}")

In [ ]:
pdf_processor = PDFProcessor()
chunker = TextChunker(chunk_size=600, chunk_overlap=100)  # Smaller chunks = faster
vector_store = VectorStore()
print("✅ Components initialized")

In [ ]:
print(f"🤖 Loading {MODEL_CHOICE.upper()} model...\n")

if MODEL_CHOICE == 'flan-t5':
    # FAST OPTION: Flan-T5 Large (1-2 seconds per query)
    model_name = 'google/flan-t5-large'
    print("   Model: Flan-T5 Large (780M params)")
    print("   Download: ~1GB")
    print("   Speed: ⚡⚡⚡ Ultra-fast (1-2 sec)\n")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16  # Half precision for speed
    )
    
    llm_pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=150,  # Short, fast responses
        temperature=0.7,
        do_sample=True
    )
    
    def generate(prompt, context):
        # Optimized prompt for Flan-T5 - simpler is better!
        full_prompt = f"answer: {prompt}\n\ncontext: {context[:600]}"
        result = llm_pipe(
            full_prompt, 
            max_new_tokens=80,
            do_sample=False,  # Greedy decoding = faster and more consistent
            num_beams=1
        )[0]['generated_text']
        return result.strip()

else:
    # BETTER QUALITY: Mistral-7B (3-5 seconds per query)
    from transformers import BitsAndBytesConfig
    
    model_name = 'mistralai/Mistral-7B-Instruct-v0.2'
    print("   Model: Mistral-7B-Instruct")
    print("   Download: ~4GB")
    print("   Speed: ⚡⚡ Fast (3-5 sec)\n")
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    
    llm_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=200,
        temperature=0.7
    )
    
    def generate(prompt, context):
        full_prompt = f"[INST] Answer concisely based on context.\n\nContext: {context[:1500]}\n\nQuestion: {prompt}\n\nAnswer: [/INST]"
        result = llm_pipe(full_prompt, max_new_tokens=200, do_sample=True)[0]['generated_text']
        return result.split('[/INST]')[-1].strip()

print("✅ LLM ready!")
if torch.cuda.is_available():
    print(f"   GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f}GB")

In [ ]:
class RAGSystem:
    def __init__(self, pdf_proc, chunker, emb_model, vec_store, gen_fn):
        self.pdf_proc = pdf_proc
        self.chunker = chunker
        self.emb_model = emb_model
        self.vec_store = vec_store
        self.generate = gen_fn
    
    def ingest_pdf(self, pdf_path: str) -> Dict:
        pdf_data = self.pdf_proc.extract_text(pdf_path)
        if not pdf_data:
            return {'error': 'Failed'}
        
        chunks = self.chunker.chunk_with_metadata(pdf_data['text_by_page'], pdf_data['file_name'])
        print(f"   {len(chunks)} chunks → ", end="", flush=True)
        
        embeddings = self.emb_model.encode(
            [c['text'] for c in chunks],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        print("embedded → ", end="", flush=True)
        
        self.vec_store.add_documents(chunks, embeddings)
        print(f"stored ✓")
        
        return {'file_name': pdf_data['file_name'], 'num_pages': pdf_data['num_pages'], 'num_chunks': len(chunks)}
    
    def query(self, question: str, top_k=3) -> Dict:
        print(f"🔍 ", end="", flush=True)
        
        # 1. Embed query
        start = time.time()
        query_emb = self.emb_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
        print(f"search → ", end="", flush=True)
        
        # 2. Retrieve
        results = self.vec_store.query(query_emb, top_k)
        if not results['documents']:
            return {'answer': 'No docs found', 'sources': [], 'time': 0}
        print(f"{len(results['documents'])} chunks → ", end="", flush=True)
        
        # 3. Format context
        context = '\n'.join([f"[Pg{m['page']}] {doc}" for doc, m in zip(results['documents'], results['metadatas'])])
        
        # 4. Generate
        print(f"generate → ", end="", flush=True)
        answer = self.generate(question, context)
        
        elapsed = time.time() - start
        print(f"done ✓")
        
        return {
            'answer': answer,
            'sources': results['metadatas'],
            'time': elapsed
        }

rag = RAGSystem(pdf_processor, chunker, embedding_model, vector_store, generate)
print("\n🎉 RAG System Ready!")

In [ ]:
import urllib.request

os.makedirs('/kaggle/working/pdfs', exist_ok=True)
print("📥 Downloading sample PDF...")

try:
    urllib.request.urlretrieve(
        "https://arxiv.org/pdf/1706.03762.pdf",
        "/kaggle/working/pdfs/attention.pdf"
    )
    pdf_files = ['/kaggle/working/pdfs/attention.pdf']
    print("✅ Downloaded: attention.pdf")
except:
    print("❌ Download failed")
    pdf_files = []

# Look for uploaded PDFs
if os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        pdf_files.extend([os.path.join(root, f) for f in files if f.endswith('.pdf')])

print(f"\n📚 Total PDFs: {len(pdf_files)}")

In [ ]:
if pdf_files:
    print("\n" + "="*60)
    print("📚 INGESTING PDFs")
    print("="*60 + "\n")
    
    ingested = []
    for pdf in pdf_files:
        try:
            result = rag.ingest_pdf(pdf)
            if 'error' not in result:
                ingested.append(result)
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\n✅ Ingested {len(ingested)} PDFs")
    print(f"   Total chunks: {vector_store.collection.count()}")
else:
    ingested = []

In [ ]:
if ingested:
    print("\n" + "="*60)
    print("🧪 ULTRA-FAST TEST")
    print("="*60 + "\n")
    
    q = "What is the main topic?"
    print(f"❓ {q}\n")
    
    response = rag.query(q, top_k=3)
    
    print(f"\n💡 Answer:\n{response['answer']}\n")
    
    # Fixed f-string syntax - alternating quotes
    sources = ', '.join([f"Pg{s['page']}" for s in response['sources']])
    print(f"📚 Sources: {sources}")
    print(f"⏱️  Response time: {response['time']:.1f}s")
    print("\n" + "="*60)
else:
    print("No PDFs ingested")

In [ ]:
# 💬 CUSTOM QUERY
MY_Q = "What is the transformer architecture?"  # Edit here

if ingested:
    print(f"❓ {MY_Q}\n")
    r = rag.query(MY_Q, top_k=3)
    print(f"💡 Answer:\n{r['answer']}\n")
    
    # Sources display
    sources_list = ', '.join([f"Pg{s['page']}" for s in r['sources']])
    print(f"📚 Sources: {sources_list}")
    print(f"⏱️  {r['time']:.1f}s")
else:
    print("Ingest PDFs first")

In [ ]:
def answer_q(question, top_k):
    if not question.strip():
        return "Enter a question", ""
    try:
        r = rag.query(question, top_k=int(top_k))
        sources = "\n".join([f"[{i+1}] Pg{s['page']} - {s['file_name']}" for i, s in enumerate(r['sources'])])
        return f"{r['answer']}\n\n⏱️ {r['time']:.1f}s", sources
    except Exception as e:
        return f"Error: {e}", ""

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# ⚡ Ultra-Fast PDF Chat\n### Using {MODEL_CHOICE.upper()} - Expect {['1-2', '3-5'][MODEL_CHOICE=='mistral']} sec responses")
    
    with gr.Row():
        with gr.Column():
            q_in = gr.Textbox(label="Question", lines=2)
            top_k = gr.Slider(1, 5, 3, step=1, label="Sources")
            btn = gr.Button("Ask", variant="primary")
        with gr.Column():
            ans = gr.Textbox(label="Answer", lines=8)
            src = gr.Textbox(label="Sources", lines=3)
    
    btn.click(answer_q, inputs=[q_in, top_k], outputs=[ans, src])
    gr.Examples([["What is this about?"], ["Summarize."], ["What is the methodology?"]], inputs=q_in)

print("🚀 Launching Gradio...\n")
demo.launch(share=True, debug=False)